# AIC 2026 — OCR keyframes bằng OCR.space API (Colab + Kaggle)

Notebook tự nhận diện Google Colab hoặc Kaggle, quét keyframe, gọi [OCR.space API](https://ocr.space/ocrapi) và lưu JSON theo từng video. **Không dùng EasyOCR hoặc VietOCR.**

- **Colab:** đọc và ghi trên Google Drive như cấu hình cũ.
- **Kaggle:** đọc dataset tại `/kaggle/input/datasets/fatle542/aic-dataset` và ghi kết quả vào `/kaggle/working/OCR_OCRSpace`.

> Lưu ý: ảnh được gửi tới dịch vụ OCR.space. Không dùng notebook này cho dữ liệu nhạy cảm. Gói miễn phí có giới hạn dung lượng và số request; hãy chạy cell kiểm thử trước khi xử lý toàn bộ dataset.


In [ ]:
# Chạy được trên cả Colab và Kaggle.
# Không nâng cấp Pillow giữa runtime để tránh lỗi tương thích PIL._typing.
!pip -q install requests


In [ ]:
import os
from pathlib import Path

IS_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or Path('/kaggle/input').exists()
IS_COLAB = False

if IS_KAGGLE:
    print('Runtime: Kaggle')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        IS_COLAB = True
        print('Runtime: Google Colab')
    except ImportError:
        print('Runtime: local')


## API key

Nhập trực tiếp các OCR.space API key vào mảng trong cell bên dưới. Notebook không in key ra output.

> Lưu ý: nếu chia sẻ hoặc public notebook, hãy xóa các key khỏi cell trước.


In [ ]:
# Nhập các API key của bạn vào mảng này. Có thể dùng một hoặc nhiều key.
OCR_SPACE_KEYS_INPUT = [
    'NHAP_API_KEY_1_VAO_DAY',
    # 'NHAP_API_KEY_2_VAO_DAY',
    # 'NHAP_API_KEY_3_VAO_DAY',
]

# Chuẩn hóa, bỏ phần tử rỗng, placeholder và key trùng nhau.
OCR_SPACE_KEYS = list(dict.fromkeys(
    str(key).strip() for key in OCR_SPACE_KEYS_INPUT
    if str(key).strip() and not str(key).strip().startswith('NHAP_API_KEY_')
))
assert OCR_SPACE_KEYS, 'Hãy nhập ít nhất một API key thật vào OCR_SPACE_KEYS_INPUT.'
print(f'Đã nạp {len(OCR_SPACE_KEYS)} OCR.space API key.')  # không in nội dung key


## Cấu hình môi trường và dữ liệu

Kaggle input là read-only, vì vậy kết quả và checkpoint được ghi vào `/kaggle/working/OCR_OCRSpace`. Colab vẫn dùng các đường dẫn Google Drive cũ. OCR.space yêu cầu **Engine 2** cho OCR tiếng Việt; dùng `language=auto` để tự nhận diện.


In [ ]:
if IS_KAGGLE:
    DATASET_DIRECTORY = Path('/kaggle/input/datasets/fatle542/aic-dataset')
    OCR_DIRECTORY = Path('/kaggle/working/OCR_OCRSpace')
else:
    # Giữ nguyên cấu hình Google Drive của Colab.
    DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
    OCR_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/OCR_OCRSpace')
MAP_KEYFRAMES_DIRECTORY = DATASET_DIRECTORY / 'map-keyframes-aic25-b1' / 'map-keyframes'

TARGET_FOLDERS = [
    'Keyframes_L21', 'Keyframes_L22', 'Keyframes_L23', 'Keyframes_L24',
    'Keyframes_L25', 'Keyframes_L26_a', 'Keyframes_L26_b',
    'Keyframes_L26_c', 'Keyframes_L26_d', 'Keyframes_L26_e',
    'Keyframes_L27', 'Keyframes_L28', 'Keyframes_L29', 'Keyframes_L30',
]
API_URL = 'https://api.ocr.space/parse/image'
OCR_LANGUAGE = 'auto'
OCR_ENGINE = 2
LANGUAGE_ALIASES = {'vi': 'vnm', 'vn': 'vnm', 'vie': 'vnm', 'vietnamese': 'vnm'}
OCR_LANGUAGE = LANGUAGE_ALIASES.get(OCR_LANGUAGE.strip().lower(), OCR_LANGUAGE.strip().lower())
REQUEST_TIMEOUT = 90
MAX_RETRIES = 4
REQUEST_DELAY_SECONDS = 0.4
MAX_UPLOAD_BYTES = 950_000
OVERWRITE = False
MODEL_ID = f'ocr.space-engine-{OCR_ENGINE}-{OCR_LANGUAGE}'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

dataset_root = DATASET_DIRECTORY.resolve()
KEYFRAME_ROOTS, missing_roots = [], []
for folder in TARGET_FOLDERS:
    relative = Path(folder.strip())
    assert not relative.is_absolute(), f'Chỉ nhận đường dẫn tương đối: {folder}'
    selected = (dataset_root / relative).resolve()
    assert selected == dataset_root or dataset_root in selected.parents
    (KEYFRAME_ROOTS if selected.is_dir() else missing_roots).append(selected if selected.is_dir() else folder)
assert KEYFRAME_ROOTS, 'Không tìm thấy thư mục keyframe nào'
assert MAP_KEYFRAMES_DIRECTORY.is_dir(), f'Thiếu map-keyframes: {MAP_KEYFRAMES_DIRECTORY}'
OUTPUT_ROOT = OCR_DIRECTORY.resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if missing_roots:
    print('Bỏ qua thư mục không tồn tại:', ', '.join(map(str, missing_roots)))
print('Input roots:', len(KEYFRAME_ROOTS), '| Output:', OUTPUT_ROOT, '| Model:', MODEL_ID)

In [ ]:
import csv
import re

VIDEO_ID_PATTERN = re.compile(r'^L\d{2}_V\d{3}$')

def find_video_dirs(root):
    base = root / 'keyframes'
    if not base.is_dir():
        base = root
    return [p for p in sorted(base.iterdir()) if p.is_dir() and VIDEO_ID_PATTERN.match(p.name)]

def load_keyframe_map(video_id):
    csv_path = MAP_KEYFRAMES_DIRECTORY / f'{video_id}.csv'
    if not csv_path.is_file():
        return None
    mapping = {}
    with csv_path.open(encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            try:
                mapping[int(row['n'])] = {
                    'pts_time': float(row['pts_time']),
                    'fps': float(row['fps']),
                    'frame_idx': int(row['frame_idx']),
                }
            except (KeyError, TypeError, ValueError):
                continue
    return mapping or None

def keyframe_order(path):
    return int(path.stem) if path.stem.isdigit() else None

video_dirs = sorted((v for root in KEYFRAME_ROOTS for v in find_video_dirs(root)), key=lambda p: p.name)
print(f'Tìm thấy {len(video_dirs)} video; {sum(load_keyframe_map(v.name) is not None for v in video_dirs)} video có map')

## OCR.space client

Client gửi ảnh bằng multipart POST, yêu cầu overlay để lấy bounding box từng từ, kiểm tra cả HTTP status lẫn lỗi trong JSON, và retry với exponential backoff cho lỗi tạm thời. OCR.space không trả confidence cho từng từ, vì vậy trường `confidence` được đặt là `null`.

In [ ]:
import io
import time
import requests
from PIL import Image, ImageOps

session = requests.Session()
active_key_index = 0

def prepare_image(image_path, max_bytes=MAX_UPLOAD_BYTES):
    image = ImageOps.exif_transpose(Image.open(image_path)).convert('RGB')
    quality = 90
    while True:
        buffer = io.BytesIO()
        image.save(buffer, format='JPEG', quality=quality, optimize=True)
        if buffer.tell() <= max_bytes:
            return buffer.getvalue(), image.size
        if quality > 55:
            quality -= 10
        else:
            width, height = image.size
            if min(width, height) <= 320:
                raise ValueError(f'Không thể nén {image_path.name} xuống dưới {max_bytes} bytes')
            image.thumbnail((int(width * 0.85), int(height * 0.85)), Image.Resampling.LANCZOS)
            quality = 80

def error_text(payload):
    value = payload.get('ErrorMessage') or payload.get('ErrorDetails') or 'OCR.space báo lỗi không xác định'
    return '; '.join(map(str, value)) if isinstance(value, list) else str(value)

def call_ocr_space(image_path):
    global active_key_index
    image_bytes, uploaded_size = prepare_image(image_path)
    form = {
        'language': OCR_LANGUAGE,
        'isOverlayRequired': 'true',
        'detectOrientation': 'true',
        'scale': 'true',
        'OCREngine': str(OCR_ENGINE),
    }
    request_forms = [form]
    if OCR_LANGUAGE != 'auto':
        request_forms.append({**form, 'language': 'auto', 'OCREngine': '2'})
    attempts = max(MAX_RETRIES + 1, len(OCR_SPACE_KEYS))
    last_error = None
    for request_form in request_forms:
        language_invalid = False
        for attempt in range(attempts):
            key_index = (active_key_index + attempt) % len(OCR_SPACE_KEYS)
            try:
                response = session.post(
                    API_URL, headers={'apikey': OCR_SPACE_KEYS[key_index]}, data=request_form,
                    files={'file': (image_path.stem + '.jpg', image_bytes, 'image/jpeg')},
                    timeout=REQUEST_TIMEOUT,
                )
                if response.status_code == 429 or response.status_code >= 500:
                    raise requests.HTTPError(f'HTTP {response.status_code}', response=response)
                response.raise_for_status()
                payload = response.json()
                if payload.get('IsErroredOnProcessing') or int(payload.get('OCRExitCode', 0)) not in (1, 2):
                    message = error_text(payload)
                    if 'e201' in message.lower():
                        last_error = RuntimeError(message)
                        language_invalid = True
                        print(f"      language={request_form['language']} không hợp lệ; thử chế độ fallback")
                        break
                    if any(token in message.lower() for token in ('quota', 'limit', 'rate', 'api key')):
                        raise requests.HTTPError(message, response=response)
                    raise RuntimeError(message)
                active_key_index = key_index
                return payload, uploaded_size, len(image_bytes)
            except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as exc:
                last_error = exc
                if attempt >= attempts - 1:
                    break
                wait = min(30, 2 ** attempt)
                next_index = (key_index + 1) % len(OCR_SPACE_KEYS)
                print(f'      key #{key_index + 1} lỗi tạm thời ({exc}); chuyển key #{next_index + 1} sau {wait}s')
                time.sleep(wait)
        if language_invalid:
            continue
    raise last_error or RuntimeError('OCR.space thất bại')

def parse_ocr_response(payload):
    detections, texts = [], []
    for page in payload.get('ParsedResults') or []:
        parsed_text = (page.get('ParsedText') or '').strip()
        if parsed_text:
            texts.append(parsed_text)
        overlay = page.get('TextOverlay') or {}
        for line in overlay.get('Lines') or []:
            for word in line.get('Words') or []:
                text = (word.get('WordText') or '').strip()
                if not text:
                    continue
                left, top = int(word.get('Left', 0)), int(word.get('Top', 0))
                width, height = int(word.get('Width', 0)), int(word.get('Height', 0))
                detections.append({
                    'text': text, 'confidence': None,
                    'box': [left, top, left + width, top + height],
                })
    return '\n'.join(texts).strip(), detections

def ocr_keyframe(image_path):
    payload, uploaded_size, upload_bytes = call_ocr_space(image_path)
    text, detections = parse_ocr_response(payload)
    return {
        'text': text, 'detections': detections,
        'uploaded_size': list(uploaded_size), 'upload_bytes': upload_bytes,
        'processing_ms': payload.get('ProcessingTimeInMilliseconds'),
    }

## Kiểm thử ít ảnh trước

Cell mặc định chỉ dùng tối đa 5 request. Hãy kiểm tra chất lượng và quota trước khi chạy toàn bộ.

In [ ]:
TEST_REQUESTS = 5  #@param {type:'integer'}
assert video_dirs, 'Không tìm thấy video nào'
sample_images = sorted(
    (p for p in video_dirs[0].iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
    key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name),
)[:max(1, TEST_REQUESTS)]
for image_path in sample_images:
    result = ocr_keyframe(image_path)
    print(image_path.name, '->', result['text'][:200].replace('\n', ' | ') or '(không có chữ)')
    time.sleep(REQUEST_DELAY_SECONDS)

## Chạy theo video, có checkpoint

File `.partial.json` được cập nhật sau từng keyframe. Nếu Colab ngắt kết nối, chạy lại notebook sẽ tiếp tục từ ảnh kế tiếp. Khi video hoàn tất, file được đổi thành `.json`.

In [ ]:
import json

def output_json_path(video_id):
    return OUTPUT_ROOT / f'{video_id}.json'

def partial_json_path(video_id):
    return OUTPUT_ROOT / f'{video_id}.partial.json'

def new_payload(video_dir, keyframe_map):
    return {
        'video_id': video_dir.name, 'source': str(video_dir), 'model': MODEL_ID,
        'language': OCR_LANGUAGE, 'ocr_engine': OCR_ENGINE,
        'has_keyframe_map': keyframe_map is not None, 'complete': False,
        'keyframe_count': 0, 'keyframes': [],
    }

def atomic_write_json(path, payload):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def ocr_video(video_dir, progress_every=25):
    video_id = video_dir.name
    keyframe_map = load_keyframe_map(video_id)
    images = sorted(
        (p for p in video_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS),
        key=lambda p: (keyframe_order(p) is None, keyframe_order(p) or 0, p.name),
    )
    partial = partial_json_path(video_id)
    if partial.exists() and not OVERWRITE:
        payload = json.loads(partial.read_text(encoding='utf-8'))
        if payload.get('model') != MODEL_ID:
            payload = new_payload(video_dir, keyframe_map)
    else:
        payload = new_payload(video_dir, keyframe_map)
    completed = {item['keyframe'] for item in payload['keyframes']}
    for index, image_path in enumerate(images, 1):
        if image_path.name in completed:
            continue
        order = keyframe_order(image_path)
        mapped = keyframe_map.get(order) if keyframe_map and order is not None else None
        result = ocr_keyframe(image_path)
        payload['keyframes'].append({
            'keyframe': image_path.name, 'n': order,
            'frame_idx': mapped['frame_idx'] if mapped else None,
            'pts_time': mapped['pts_time'] if mapped else None,
            'fps': mapped['fps'] if mapped else None,
            **result,
        })
        payload['keyframe_count'] = len(payload['keyframes'])
        atomic_write_json(partial, payload)
        if progress_every and index % progress_every == 0:
            print(f'    {index}/{len(images)} keyframe')
        time.sleep(REQUEST_DELAY_SECONDS)
    payload['complete'] = True
    payload['keyframe_count'] = len(payload['keyframes'])
    destination = output_json_path(video_id)
    atomic_write_json(destination, payload)
    if partial.exists():
        partial.unlink()
    return payload, destination

In [ ]:
import traceback

MAX_VIDEOS = 0  #@param {type:'integer'}
selected_videos = video_dirs[:MAX_VIDEOS] if MAX_VIDEOS > 0 else video_dirs
success = skipped = failed = 0
failed_videos = []
for index, video_dir in enumerate(selected_videos, 1):
    destination = output_json_path(video_dir.name)
    if destination.exists() and not OVERWRITE:
        try:
            existing = json.loads(destination.read_text(encoding='utf-8'))
        except Exception:
            existing = {}
        if existing.get('model') == MODEL_ID and existing.get('complete'):
            skipped += 1
            print(f'[{index}/{len(selected_videos)}] SKIP {video_dir.name}')
            continue
    print(f'[{index}/{len(selected_videos)}] OCR  {video_dir.name}')
    try:
        payload, _ = ocr_video(video_dir)
        print(f"    {sum(bool(k['text']) for k in payload['keyframes'])}/{payload['keyframe_count']} ảnh có chữ")
        success += 1
    except Exception as exc:
        failed += 1
        failed_videos.append({'video_id': video_dir.name, 'error': repr(exc)})
        traceback.print_exc()
atomic_write_json(OUTPUT_ROOT / '_failed.json', failed_videos)
print(f'Hoàn tất: success={success}, skipped={skipped}, failed={failed}')

## Xuất JSONL để index

In [ ]:
output_lines = no_frame_idx = 0
jsonl_path = OUTPUT_ROOT / 'ocr_index.jsonl'
with jsonl_path.open('w', encoding='utf-8') as handle:
    for json_path in sorted(OUTPUT_ROOT.glob('L*_V*.json')):
        payload = json.loads(json_path.read_text(encoding='utf-8'))
        if payload.get('model') != MODEL_ID or not payload.get('complete'):
            continue
        for keyframe in payload['keyframes']:
            if not keyframe['text']:
                continue
            no_frame_idx += keyframe['frame_idx'] is None
            row = {
                'video_id': payload['video_id'], 'frame_idx': keyframe['frame_idx'],
                'pts_time': keyframe['pts_time'], 'keyframe': keyframe['keyframe'],
                'text': keyframe['text'],
            }
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')
            output_lines += 1
print(f'Đã ghi {output_lines} dòng vào {jsonl_path}')
if no_frame_idx:
    print(f'CẢNH BÁO: {no_frame_idx} dòng thiếu frame_idx')